# Módulo 1: Radiografía del Sueldo, Presupuesto Personal y Gastos Hormiga
**Responsable OED:** Nidia Velásquez (*Ingeniera Mecánica — Especialista en Presupuestos*)  
**Sistema:** Asesor Financiero e Inversiones OED (Colombia — 100% Local)

---

## Enfoque de Ingeniería: Balance Termodinámico de Flujo de Caja
Desde la ingeniería mecánica, las finanzas personales operan bajo la **Ley de Conservación de Flujo**:

$$\text{Flujo Neto Disponible (IDN)} = \text{Ingreso Bruto} + \text{Auxilios} - \text{Pérdidas de Carga Legales (Salud, Pensión, FSP, ARL, Retefuente)}$$

$$\text{Excedente Libre Inversible} = \text{IDN} - \big(\text{Costos Fijos Vitales} + \text{Estilo de Vida} + \underbrace{\text{Gastos Hormiga + 4x1000}}_{\text{Fugas de Presión}} + \text{Cuotas de Deuda}\big)$$

> **Nota Regulatoria (SFC Colombia):** Herramienta pedagógica y analítica local de planeación presupuestal basada en la Ley 100 de 1993, Ley 797 de 2003, Reforma Tributaria Ley 2277 de 2022 (Art. 383 y 336 E.T.) y parámetros vigentes en `config/config.yaml`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

# Asegurar importación limpia desde src/ sin importar desde qué carpeta se ejecute Jupyter
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.finance import cargar_configuracion
from src.mod1_presupuesto import (
    comparar_modalidades_laborales,
    crear_figura_costo_oportunidad_hormiga,
    crear_figura_sankey_presupuesto,
    crear_figura_waterfall_sueldo,
    cuantificar_gastos_hormiga_y_fugas,
    diagnosticar_presupuesto_y_flujo_caja,
    ejecutar_modulo_1_perfil_usuario,
    liquidar_ingreso_neto_colombia,
)

pd.options.display.float_format = "${:,.0f}".format
config = cargar_configuracion()
perfil_semilla = config["perfil_semilla_taller"]
print("Parámetros Macro Cargados (Colombia):")
print(f"  • SMMLV: ${config['parametros_macro_colombia']['smmlv_cop']:,.0f} COP")
print(f"  • Auxilio de Transporte: ${config['parametros_macro_colombia']['auxilio_transporte_cop']:,.0f} COP")
print(f"  • UVT DIAN: ${config['parametros_macro_colombia']['uvt_cop']:,.0f} COP")

---
## 1. Radiografía del Sueldo Colombiano (Nómina vs. Integral vs. Independiente OPS)
Evaluamos primero el caso maestro del taller (**Salario Bruto de `$6.500.000 COP`**) y luego comparamos las **3 modalidades contractuales colombianas** y el efecto de ahorro tributario de una **Cuenta AFC / Pensión Voluntaria**.

In [ ]:
nomina_semilla = liquidar_ingreso_neto_colombia(
    ingreso_bruto_mensual_cop=perfil_semilla["ingreso_bruto_mensual_cop"],
    modalidad_laboral=perfil_semilla["modalidad_laboral"],
    config=config,
)

resumen_nomina = pd.Series(
    {
        "Salario Bruto Mensual": nomina_semilla["ingreso_bruto_mensual_cop"],
        "Auxilio de Transporte (<= 2 SMMLV)": nomina_semilla["auxilio_transporte_cop"],
        "IBC Seguridad Social": nomina_semilla["ibc_cotizacion_cop"],
        "(-) Salud Obligatoria (4%)": -nomina_semilla["salud_obligatoria_cop"],
        "(-) Pensión Obligatoria (4%)": -nomina_semilla["pension_obligatoria_cop"],
        "(-) Fondo Solidaridad Pensional (>= 4 SMMLV)": -nomina_semilla["fondo_solidaridad_pensional_cop"],
        "(-) Retención en la Fuente (Art. 383 E.T.)": -nomina_semilla["retencion_fuente_cop"],
        "(=) Ingreso Disponible Neto (IDN)": nomina_semilla["ingreso_neto_mensual_cop"],
        "(+) Provisión Prima Mensualizada (Informativa)": nomina_semilla["provisiones_prestacionales_informativas"][
            "prima_servicios_mensualizada_cop"
        ],
        "(+) Provisión Cesantías + Int. (Informativa)": nomina_semilla["provisiones_prestacionales_informativas"][
            "cesantias_mensualizadas_cop"
        ]
        + nomina_semilla["provisiones_prestacionales_informativas"]["intereses_cesantias_mensualizados_cop"],
    },
    name="Monto (COP)",
).to_frame()

display(resumen_nomina)

# Comparativa entre Empleado, Salario Integral e Independiente (OPS con IBC 40%)
comparativa = comparar_modalidades_laborales(
    ingreso_bruto_mensual_cop=19_000_000.0,
    aporte_voluntario_afc_fpv_cop=1_500_000.0,
    config=config,
)
df_comp = pd.DataFrame(comparativa)[
    [
        "modalidad_laboral",
        "ingreso_bruto_mensual_cop",
        "ibc_cotizacion_cop",
        "salud_obligatoria_cop",
        "pension_obligatoria_cop",
        "fondo_solidaridad_pensional_cop",
        "arl_cop",
        "retencion_fuente_cop",
        "ahorro_tributario_afc_mensual_cop",
        "ingreso_neto_mensual_cop",
    ]
]
print("\nComparativa Contractual para $19.000.000 COP Brutos (con aporte AFC de $1.500.000 COP):")
display(df_comp)

---
## 2. Detección de "Fugas de Presión": Gastos Hormiga, 4x1000 y Costo de Oportunidad
Los pequeños gastos cotidianos y los costos financieros evitables (cuotas de manejo o cuentas sin marcar como exentas del **4x1000**) drenan silenciosamente el patrimonio. Cuantificamos cuánto capital tendría el usuario en **5 y 10 años** si invirtiera esas fugas al **10% EA**.

In [ ]:
detalle_hormiga_demo = [
    {"concepto": "Café / snacks fuera de casa", "monto_unitario_cop": 6_500, "frecuencia_mensual": 20},
    {"concepto": "Recargos y propinas de domicilios", "monto_unitario_cop": 12_500, "frecuencia_mensual": 8},
    {"concepto": "Suscripciones digitales subutilizadas", "monto_unitario_cop": 45_000, "frecuencia_mensual": 1},
    {"concepto": "Compras impulsivas / antojos fin de semana", "monto_unitario_cop": 22_500, "frecuencia_mensual": 2},
]  # Suma exacta = $320.000 COP/mes (perfil semilla OED)

fugas = cuantificar_gastos_hormiga_y_fugas(
    detalle_gastos_hormiga=detalle_hormiga_demo,
    cuotas_manejo_mensual_cop=28_000.0,
    monto_movimientos_gmf_mensual_cop=3_500_000.0,
    cuenta_exenta_4x1000=False,  # Ejemplo de cuenta secundaria NO marcada como exenta de 4x1000
    ingreso_neto_mensual_cop=nomina_semilla["ingreso_neto_mensual_cop"],
    tasa_oportunidad_ea=0.10,
    config=config,
)

display(pd.DataFrame(fugas["detalle_items"]))
print(f"• Gastos Hormiga Mensuales:                ${fugas['gastos_hormiga_mensual_cop']:,.0f} COP")
print(f"• Fricción Bancaria (4x1000 + Manejo):     ${fugas['friccion_bancaria_mensual_cop']:,.0f} COP")
print(f"• FUGA TOTAL MENSUAL:                      ${fugas['fuga_total_mensual_cop']:,.0f} COP ({fugas['peso_sobre_ingreso_neto_pct']:.2f}% del IDN)")
print(f"• Costo de Oportunidad Hormiga (5 años):   ${fugas['costo_oportunidad_hormiga_5a_cop']:,.0f} COP")
print(f"• Costo de Oportunidad Hormiga (10 años):  ${fugas['costo_oportunidad_hormiga_10a_cop']:,.0f} COP")
print(f"• Costo de Oportunidad Fuga Total (10a):   ${fugas['costo_oportunidad_fuga_total_10a_nominal_cop']:,.0f} COP (Real constante: ${fugas['costo_oportunidad_fuga_total_10a_real_cop']:,.0f} COP)")

---
## 3. Diagnóstico Integral `CU-01`, Regla 50/30/20 y Diagramas Interactivos (Sankey & Waterfall)
Ejecutamos el diagnóstico completo del Caso OED y visualizamos el recorrido de cada peso desde el salario bruto hasta el `excedente_mensual_cop`.

In [ ]:
diagnostico_oed = diagnosticar_presupuesto_y_flujo_caja(
    ingreso_bruto_mensual_cop=perfil_semilla["ingreso_bruto_mensual_cop"],
    gastos_fijos_vitales_cop=perfil_semilla["gastos_fijos_vitales_cop"],
    gastos_variables_estilo_vida_cop=perfil_semilla["gastos_variables_estilo_vida_cop"],
    gastos_hormiga_mensual_cop=perfil_semilla["gastos_hormiga_mensual_cop"],
    cuotas_deuda_mensual_cop=600_000.0,
    liquidez_actual_fondo_emergencia_cop=perfil_semilla["liquidez_actual_fondo_emergencia_cop"],
    modalidad_laboral=perfil_semilla["modalidad_laboral"],
    detalle_gastos_hormiga=detalle_hormiga_demo,
    config=config,
)

b50 = diagnostico_oed["balance_50_30_20"]
df_50_30_20 = pd.DataFrame(
    [
        {
            "Bloque Presupuestal": "50% Necesidades Vitales",
            "Monto Real (COP)": b50["necesidades_50"]["real_cop"],
            "Meta 50/30/20 (COP)": b50["necesidades_50"]["meta_cop"],
            "% Real": f"{b50['necesidades_50']['real_pct']:.1f}%",
            "Cumple": "Sí" if b50["necesidades_50"]["cumple"] else "Ajustar",
        },
        {
            "Bloque Presupuestal": "30% Estilo de Vida + Fugas",
            "Monto Real (COP)": b50["estilo_vida_30"]["real_cop"],
            "Meta 50/30/20 (COP)": b50["estilo_vida_30"]["meta_cop"],
            "% Real": f"{b50['estilo_vida_30']['real_pct']:.1f}%",
            "Cumple": "Sí" if b50["estilo_vida_30"]["cumple"] else "Ajustar",
        },
        {
            "Bloque Presupuestal": "20% Excedente Libre (Ahorro / Inversión)",
            "Monto Real (COP)": b50["ahorro_patrimonio_20"]["real_cop"],
            "Meta 50/30/20 (COP)": b50["ahorro_patrimonio_20"]["meta_cop"],
            "% Real": f"{b50['ahorro_patrimonio_20']['real_pct']:.1f}%",
            "Cumple": "Sí" if b50["ahorro_patrimonio_20"]["cumple"] else "Ajustar",
        },
    ]
)
display(df_50_30_20)
print("Estado del Flujo de Caja:", diagnostico_oed["estado_flujo_caja"])
print("Recomendación Ejecutiva:", diagnostico_oed["plan_accion_flujo_caja"]["recomendacion_ejecutiva"])

In [ ]:
fig_sankey = crear_figura_sankey_presupuesto(diagnostico_oed)
fig_sankey.show()

In [ ]:
fig_waterfall = crear_figura_waterfall_sueldo(diagnostico_oed)
fig_waterfall.show()

fig_oportunidad = crear_figura_costo_oportunidad_hormiga(diagnostico_oed)
fig_oportunidad.show()

---
## 4. Contrato de Integración OED (Entrega a Módulos 2, 3, 4 y 5)
Verificamos la función `ejecutar_modulo_1_perfil_usuario()` que alimenta directamente el pipeline del equipo OED con las llaves pactadas en `PLAN_EJECUCION_TALLER_OED.md`.

In [ ]:
perfil_actualizado = ejecutar_modulo_1_perfil_usuario(perfil_semilla, config=config)

llaves_contrato_mod1 = [
    "ingreso_neto_mensual_cop",
    "excedente_mensual_cop",
    "meses_fondo_emergencia",
    "costo_oportunidad_hormiga_10a_cop",
]
print("Contrato de Salida del Módulo 1 (Nidia Velásquez):")
for k in llaves_contrato_mod1:
    val = perfil_actualizado[k]
    if "cop" in k:
        print(f"  • {k}: ${val:,.2f} COP")
    else:
        print(f"  • {k}: {val:.2f} meses")